## File traces5000.bin
contains 5.000 power traces
Each power trace consists of 2.000 samples. The file therefore contains 2,000 x 5,000 power samples.
Every power sample is a signed 16-bit integer (int16_t, signed short). The file is therefore 2 x 2,000 x 5,000 = 20,000,000 bytes.

## File ciphertext5000.bin
contains 5.000 ciphertexts. Each ciphertext consists of 16 bytes. The file therefore contains 16 x 5,000 bytes.

little-endian

In every file, first, there are 2,000 samples (or 16 bytes of ciphertext respectively) collected during the first encryption. After that, there are 2,000 samples (or 16 bytes of ciphertext respectively) collected during the second encryption. And so on.

In [ ]:
from pathlib import Path
import numpy as np
# 5000 encryptions
# each 2000 traces
# each 16 Byte ciphertexts

data_dir: Path = Path.cwd() / 'ni-hsc-lab-02-data'
ptrace_length = 2000 # of 16 bit samples

ptraces = np.fromfile(data_dir / 'traces5000.bin', dtype=np.int16).reshape(-1, ptrace_length)
print(f'Power traces shape: {ptraces.shape}')

ct_length = 16 # Bytes
# load ciphertexts as unsigned bytes so values are in 0..255 (avoids negative indices when indexing SBoxInverse)
ciphertexts = np.fromfile(data_dir / 'ciphertext5000.bin', dtype=np.uint8).reshape(-1, ct_length)
print(f'Ciphertext shape: {ciphertexts.shape}')

assert ptraces.shape[0] == ciphertexts.shape[0]

# 2. Perform DPA attack (5 b)

Perform DPA and multi-bit DPA attacks on a single byte of key using the prepared data.

## Single-bit DPA

1. Choose a byte of key (subkey) to attack on. -> 1st byte

2. Choose following intermediate values:
- v1: result of the last but one round
- v2: ciphertext
- leakage function: L̂(v1,v2)=LSB(v1⊕v2).

3. Compute intermediate values for each of possible subkey values.
4. Divide power traces into two sets according to the leakage function.
5. Choose the most probable key candidate based on differences of means (using, e.g., maximum of DoM values).

6. Verify your choice using a graph displaying traces of differences of means for all of the key candidates highlighting the one you have chosen.


In [ ]:
ShiftRow = np.array([ 0, 5, 10, 15, 4, 9, 14, 3, 8, 13, 2, 7, 12, 1, 6, 11 ], dtype=np.uint8)
ShiftRowInverse = np.array([
    0, 13, 10, 7,   # row 0 → no shift
    4, 1, 14, 11,   # row 1 → shift right by 1
    8, 5, 2, 15,    # row 2 → shift right by 2
    12, 9, 6, 3     # row 3 → shift right by 3
], dtype=np.uint8)

SBoxInverse = np.array(
            [0x52 ,0x09 ,0x6A ,0xD5 ,0x30 ,0x36 ,0xA5 ,0x38 ,0xBF ,0x40 ,0xA3 ,0x9E ,0x81 ,0xF3 ,0xD7 ,0xFB
            ,0x7C ,0xE3 ,0x39 ,0x82 ,0x9B ,0x2F ,0xFF ,0x87 ,0x34 ,0x8E ,0x43 ,0x44 ,0xC4 ,0xDE ,0xE9 ,0xCB
            ,0x54 ,0x7B ,0x94 ,0x32 ,0xA6 ,0xC2 ,0x23 ,0x3D ,0xEE ,0x4C ,0x95 ,0x0B ,0x42 ,0xFA ,0xC3 ,0x4E
            ,0x08 ,0x2E ,0xA1 ,0x66 ,0x28 ,0xD9 ,0x24 ,0xB2 ,0x76 ,0x5B ,0xA2 ,0x49 ,0x6D ,0x8B ,0xD1 ,0x25
            ,0x72 ,0xF8 ,0xF6 ,0x64 ,0x86 ,0x68 ,0x98 ,0x16 ,0xD4 ,0xA4 ,0x5C ,0xCC ,0x5D ,0x65 ,0xB6 ,0x92
            ,0x6C ,0x70 ,0x48 ,0x50 ,0xFD ,0xED ,0xB9 ,0xDA ,0x5E ,0x15 ,0x46 ,0x57 ,0xA7 ,0x8D ,0x9D ,0x84
            ,0x90 ,0xD8 ,0xAB ,0x00 ,0x8C ,0xBC ,0xD3 ,0x0A ,0xF7 ,0xE4 ,0x58 ,0x05 ,0xB8 ,0xB3 ,0x45 ,0x06
            ,0xD0 ,0x2C ,0x1E ,0x8F ,0xCA ,0x3F ,0x0F ,0x02 ,0xC1 ,0xAF ,0xBD ,0x03 ,0x01 ,0x13 ,0x8A ,0x6B
            ,0x3A ,0x91 ,0x11 ,0x41 ,0x4F ,0x67 ,0xDC ,0xEA ,0x97 ,0xF2 ,0xCF ,0xCE ,0xF0 ,0xB4 ,0xE6 ,0x73
            ,0x96 ,0xAC ,0x74 ,0x22 ,0xE7 ,0xAD ,0x35 ,0x85 ,0xE2 ,0xF9 ,0x37 ,0xE8 ,0x1C ,0x75 ,0xDF ,0x6E
            ,0x47 ,0xF1 ,0x1A ,0x71 ,0x1D ,0x29 ,0xC5 ,0x89 ,0x6F ,0xB7 ,0x62 ,0x0E ,0xAA ,0x18 ,0xBE ,0x1B
            ,0xFC ,0x56 ,0x3E ,0x4B ,0xC6 ,0xD2 ,0x79 ,0x20 ,0x9A ,0xDB ,0xC0 ,0xFE ,0x78 ,0xCD ,0x5A ,0xF4
            ,0x1F ,0xDD ,0xA8 ,0x33 ,0x88 ,0x07 ,0xC7 ,0x31 ,0xB1 ,0x12 ,0x10 ,0x59 ,0x27 ,0x80 ,0xEC ,0x5F
            ,0x60 ,0x51 ,0x7F ,0xA9 ,0x19 ,0xB5 ,0x4A ,0x0D ,0x2D ,0xE5 ,0x7A ,0x9F ,0x93 ,0xC9 ,0x9C ,0xEF
            ,0xA0 ,0xE0 ,0x3B ,0x4D ,0xAE ,0x2A ,0xF5 ,0xB0 ,0xC8 ,0xEB ,0xBB ,0x3C ,0x83 ,0x53 ,0x99 ,0x61
            ,0x17 ,0x2B ,0x04 ,0x7E ,0xBA ,0x77 ,0xD6 ,0x26 ,0xE1 ,0x69 ,0x14 ,0x63 ,0x55 ,0x21 ,0x0C ,0x7D],
            dtype=np.uint8)

In [ ]:
def calc_intermediate_values(ct, key_guess, ct_byte_idx):
    """
    :param ct: A single ciphertext.
    :param key_guess: Guess of a single key byte.
    :param ct_byte_idx: Attacked byte indexof the ciphertext.
    :return: Hypothesis of state register after 9. round and state register after 10. round.
    """
    post_round10 = ct[ct_byte_idx]

    # States after given operation. The 10th round in reverse.
    # SR shifts bytes of states, therefore perform the reverse to attack correct corresponding bytes.
    # Ex.: Byte in state register on index 1 after 10th round was byte on index 13 after 9th round.
    post_sub_bytes = ct[ShiftRowInverse[ct_byte_idx]] ^ key_guess
    post_round9 = SBoxInverse[post_sub_bytes]
    return post_round10, post_round9

In [ ]:
def calc_leakage_function(v1, v2):
    """
    Extract LSB from XOR of intermediate values.
    """
    return 0x0001 & (v1 ^ v2)

In [ ]:
def compute_doms(doms, dom_time_matrix, key_guess, l0, l1):
    """
    Differentiate leakage groups using Difference of Means and return key candidate.
    :param doms: maximal differences of means (of leakage groups) within their corresponding time series'
    :param dom_time_matrix: doms time series for each key guess
    :param l0: traces from leakage group with LSB==0
    :param l1: traces from leakage group with LSB==1
    """
    # compute per-sample mean for each group; handle empty groups safely
    if len(l0) > 0: mean0 = np.mean(np.vstack(l0), axis=0)
    else: mean0 = np.zeros(ptraces.shape[1], dtype=ptraces.dtype)

    if len(l1) > 0: mean1 = np.mean(np.vstack(l1), axis=0)
    else: mean1 = np.zeros(ptraces.shape[1], dtype=ptraces.dtype)

    dom = np.abs(mean0 - mean1)
    dom_time_matrix[key_guess] = dom
    doms[key_guess] = dom.max()



In [ ]:
# Perform single-bit DPA

# attacked byte index
key_byte_idx = 0

n_keys = 256
n_samples = ptraces.shape[1]
# container for per-key difference-of-means time series
dom_time_matrix = np.zeros((n_keys, n_samples), dtype=np.float32)
doms = np.zeros(n_keys, dtype=np.float32)   # scalar metric per key

for key_guess in range(n_keys):
    # full traces per leakage group (we need per-sample means)
    lgroup0_traces = []
    lgroup1_traces = []

    for ptrace, ct in zip(ptraces, ciphertexts):
        v1, v2 = calc_intermediate_values(ct, key_guess, key_byte_idx)
        L = calc_leakage_function(v1, v2)
        (lgroup0_traces if L == 0 else lgroup1_traces).append(ptrace)

    compute_doms(doms, dom_time_matrix, key_guess, lgroup0_traces, lgroup1_traces)

key_candidate = np.argmax(doms)
print(f'Round 10 roundkey candidate: {hex(key_candidate)}')

In [ ]:
import matplotlib.pyplot as plt
# Plot dom, highlight chosen candidate
fig, ax = plt.subplots(figsize=(12, 6))
for k in range(n_keys):
    ax.plot(dom_time_matrix[k], color='gray', alpha=0.25, linewidth=0.6)

ax.plot(dom_time_matrix[key_candidate], color='red', linewidth=2.2, label=f'chosen: {hex(key_candidate)}')
ax.set_xlabel('Sample index')
ax.set_ylabel('Absolute difference of means')
ax.set_title('Per-sample difference-of-means for all key candidates')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

Follow the procedure of single-bit DPA on the same key byte, but repeat the procedure for each of 8 bits of function L̂.

Sum up the means for every bit and evaluate the best key candidate.

Display differences of means for the best candidate for Single-bit and Multi-bit variant in a graph.

Repeat the whole procedure decreasing the number of power traces used to verify that a lower amount of data is needed for the Multi-bit variant.

In [ ]:
def calc_leakage_function_perbit(v1, v2, bit):
    """Return 0/1 for the given bit of (v1 ^ v2)."""
    return ((int(v1) ^ int(v2)) >> bit) & 1


In [ ]:
# Perform multi-bit DPA
# attacked byte index
key_byte_idx = 0

n_keys = 256
n_samples = ptraces.shape[1]
# container for per-key difference-of-means time series
dom_time_matrix_multibit = np.zeros((n_keys, n_samples), dtype=np.float32)
doms_multibit = np.zeros(n_keys, dtype=np.float32)   # scalar metric per key

for key_guess in range(n_keys):
    dom_time_sum = np.zeros(n_samples, dtype=np.float32)
    for bit in range(8):
        lgroup0_traces = []
        lgroup1_traces = []
        for ptrace, ct in zip(ptraces, ciphertexts):
            v1, v2 = calc_intermediate_values(ct, key_guess, key_byte_idx)
            L = calc_leakage_function_perbit(v1, v2, bit)
            (lgroup0_traces if L == 0 else lgroup1_traces).append(ptrace)

        if lgroup0_traces: mean0 = np.mean(np.vstack(lgroup0_traces), axis=0)
        else: mean0 = np.zeros(n_samples, dtype=ptraces.dtype)

        if lgroup1_traces: mean1 = np.mean(np.vstack(lgroup1_traces), axis=0)
        else: mean1 = np.zeros(n_samples, dtype=ptraces.dtype)

        dom_bit = np.abs(mean0 - mean1)
        dom_time_sum += dom_bit

    dom_time_matrix_multibit[key_guess] = dom_time_sum
    doms_multibit[key_guess] = dom_time_sum.max()

key_candidate_multibit = np.argmax(doms_multibit)
print(f'Round 10 roundkey candidate (multi-bit): {hex(key_candidate)}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for k in range(n_keys):
    ax.plot(dom_time_matrix_multibit[k], color='gray', alpha=0.25, linewidth=0.6)

ax.plot(dom_time_matrix[key_candidate], color='red', linewidth=2.2, label=f'chosen key byte (single-bit): {hex(key_candidate_multibit)}')
ax.plot(dom_time_matrix_multibit[key_candidate], color='green', linewidth=2.2, alpha=0.8, label=f'chosen key byte (multi-bit): {hex(key_candidate)}')
ax.set_xlabel('Sample index')
ax.set_ylabel('Summed absolute difference of means')
ax.set_title('Multi-bit DPA: summed per-bit difference-of-means for all key candidates')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

Attack each of the bytes of key one by one to obtain the whole round key.

Choose following intermediate values: v1: result of the last but one round, v2: ciphertext, and leakage function L̂(v1,v2)=HD(v1,v2)=HW(v1⊕v2).

Compute the leakage function for each pair of subkey and ciphertext.

Compute correlation matrix between the real power and the power estimated using the leakage function.

Choose the most probable key candidate (using, e.g., maximum of correlation values).

Verify your choice using a graph displaying all correlation traces highlighting the one corresponding to the chosen key candidate.

Repeat the procedure for all bytes of the key.

In [ ]:
import os
from numpy import array

class Measurement:
    """
    Class encapsulating a side-channel trace measurement, it's corresponding plaintexts
    and ciphertexts.
    :param str plaintext: path to plaintext file with hex space separated bytes
    :param str ciphertext: path to ciphertext file with hex space separated bytes
    :param str trace: path to trace file binary uint8_t samples
    :param np.array encryption_key: master key before any key scheduling occurs
    :param int key_length: key length in bytes
    """
    def __init__(self, plaintext: str, ciphertext: str,
                  trace: str, encryption_key: array = None,
                  key_length: int = 16):
        # if not os.path.isfile(plaintext):
        #     raise FileNotFoundError(f"The file '{plaintext}' was not found.")
        if not os.path.isfile(ciphertext):
            raise FileNotFoundError(f"The file '{ciphertext}' was not found.")
        if not os.path.isfile(trace):
            raise FileNotFoundError(f"The file '{trace}' was not found.")
        if encryption_key is None:
            # Guessing entropy will not be calculated in the cpa.py script
            print(f"Warning: encryption key not provided for {trace}.")
        self.plaintext_path = plaintext
        self.ciphertext_path = ciphertext
        self.trace_path = trace
        self.trace_length = ptrace_length
        # self.cnt = self.get_line_count(self.plaintext_path) # number of total measurements
        self.encryption_key = encryption_key
        self.key_length = key_length

    def get_trace_length(self) -> int:
        """
        Returns length of each trace by dividing the size of the binary file
        by the amount of measurements.
        """
        trace_size = self.get_file_size(self.trace_path)
        pt_line_count = self.get_line_count(self.plaintext_path)
        if trace_size % pt_line_count != 0:
            print(f"Trace size: {trace_size}\nPT line count: {pt_line_count}")
            raise ValueError("Binary data size is not a multiple of PT line count")
        return trace_size // pt_line_count

    def get_line_count(self, file_path: str) -> int:
        """
        Returns the amount of non-blank lines in a file.
        """
        with open(file_path, 'r') as file:
            line_count = sum(1 for line in file if line.strip())
        return line_count

    def get_file_size(self, file_path: str) -> int:
        try:
            size = os.path.getsize(file_path)
            return size
        except FileNotFoundError:
            print(f"The file '{file_path}' was not found.")
        except Exception as e:
            print(f"An error occurred: {e}")


In [ ]:
from time import time
from typing import List, Tuple
import os

import numpy as np
from colorama import Fore, Style
from aeskeyschedule import reverse_key_schedule, key_schedule

def hex_to_int(hex_str: str) -> int:
    return int(hex_str, 16)


SBox = np.array([
    0x63, 0x7C, 0x77, 0x7B, 0xF2, 0x6B, 0x6F, 0xC5, 0x30, 0x01, 0x67, 0x2B, 0xFE, 0xD7, 0xAB, 0x76,
    0xCA, 0x82, 0xC9, 0x7D, 0xFA, 0x59, 0x47, 0xF0, 0xAD, 0xD4, 0xA2, 0xAF, 0x9C, 0xA4, 0x72, 0xC0,
    0xB7, 0xFD, 0x93, 0x26, 0x36, 0x3F, 0xF7, 0xCC, 0x34, 0xA5, 0xE5, 0xF1, 0x71, 0xD8, 0x31, 0x15,
    0x04, 0xC7, 0x23, 0xC3, 0x18, 0x96, 0x05, 0x9A, 0x07, 0x12, 0x80, 0xE2, 0xEB, 0x27, 0xB2, 0x75,
    0x09, 0x83, 0x2C, 0x1A, 0x1B, 0x6E, 0x5A, 0xA0, 0x52, 0x3B, 0xD6, 0xB3, 0x29, 0xE3, 0x2F, 0x84,
    0x53, 0xD1, 0x00, 0xED, 0x20, 0xFC, 0xB1, 0x5B, 0x6A, 0xCB, 0xBE, 0x39, 0x4A, 0x4C, 0x58, 0xCF,
    0xD0, 0xEF, 0xAA, 0xFB, 0x43, 0x4D, 0x33, 0x85, 0x45, 0xF9, 0x02, 0x7F, 0x50, 0x3C, 0x9F, 0xA8,
    0x51, 0xA3, 0x40, 0x8F, 0x92, 0x9D, 0x38, 0xF5, 0xBC, 0xB6, 0xDA, 0x21, 0x10, 0xFF, 0xF3, 0xD2,
    0xCD, 0x0C, 0x13, 0xEC, 0x5F, 0x97, 0x44, 0x17, 0xC4, 0xA7, 0x7E, 0x3D, 0x64, 0x5D, 0x19, 0x73,
    0x60, 0x81, 0x4F, 0xDC, 0x22, 0x2A, 0x90, 0x88, 0x46, 0xEE, 0xB8, 0x14, 0xDE, 0x5E, 0x0B, 0xDB,
    0xE0, 0x32, 0x3A, 0x0A, 0x49, 0x06, 0x24, 0x5C, 0xC2, 0xD3, 0xAC, 0x62, 0x91, 0x95, 0xE4, 0x79,
    0xE7, 0xC8, 0x37, 0x6D, 0x8D, 0xD5, 0x4E, 0xA9, 0x6C, 0x56, 0xF4, 0xEA, 0x65, 0x7A, 0xAE, 0x08,
    0xBA, 0x78, 0x25, 0x2E, 0x1C, 0xA6, 0xB4, 0xC6, 0xE8, 0xDD, 0x74, 0x1F, 0x4B, 0xBD, 0x8B, 0x8A,
    0x70, 0x3E, 0xB5, 0x66, 0x48, 0x03, 0xF6, 0x0E, 0x61, 0x35, 0x57, 0xB9, 0x86, 0xC1, 0x1D, 0x9E,
    0xE1, 0xF8, 0x98, 0x11, 0x69, 0xD9, 0x8E, 0x94, 0x9B, 0x1E, 0x87, 0xE9, 0xCE, 0x55, 0x28, 0xDF,
    0x8C, 0xA1, 0x89, 0x0D, 0xBF, 0xE6, 0x42, 0x68, 0x41, 0x99, 0x2D, 0x0F, 0xB0, 0x54, 0xBB, 0x16
    ], dtype='uint8')


ShiftRowInverse = np.array([ 0, 5, 10, 15, 4, 9, 14, 3, 8, 13, 2, 7, 12, 1, 6, 11 ], dtype=np.uint8)
SBoxInverse = np.array(
            [0x52 ,0x09 ,0x6A ,0xD5 ,0x30 ,0x36 ,0xA5 ,0x38 ,0xBF ,0x40 ,0xA3 ,0x9E ,0x81 ,0xF3 ,0xD7 ,0xFB
            ,0x7C ,0xE3 ,0x39 ,0x82 ,0x9B ,0x2F ,0xFF ,0x87 ,0x34 ,0x8E ,0x43 ,0x44 ,0xC4 ,0xDE ,0xE9 ,0xCB
            ,0x54 ,0x7B ,0x94 ,0x32 ,0xA6 ,0xC2 ,0x23 ,0x3D ,0xEE ,0x4C ,0x95 ,0x0B ,0x42 ,0xFA ,0xC3 ,0x4E
            ,0x08 ,0x2E ,0xA1 ,0x66 ,0x28 ,0xD9 ,0x24 ,0xB2 ,0x76 ,0x5B ,0xA2 ,0x49 ,0x6D ,0x8B ,0xD1 ,0x25
            ,0x72 ,0xF8 ,0xF6 ,0x64 ,0x86 ,0x68 ,0x98 ,0x16 ,0xD4 ,0xA4 ,0x5C ,0xCC ,0x5D ,0x65 ,0xB6 ,0x92
            ,0x6C ,0x70 ,0x48 ,0x50 ,0xFD ,0xED ,0xB9 ,0xDA ,0x5E ,0x15 ,0x46 ,0x57 ,0xA7 ,0x8D ,0x9D ,0x84
            ,0x90 ,0xD8 ,0xAB ,0x00 ,0x8C ,0xBC ,0xD3 ,0x0A ,0xF7 ,0xE4 ,0x58 ,0x05 ,0xB8 ,0xB3 ,0x45 ,0x06
            ,0xD0 ,0x2C ,0x1E ,0x8F ,0xCA ,0x3F ,0x0F ,0x02 ,0xC1 ,0xAF ,0xBD ,0x03 ,0x01 ,0x13 ,0x8A ,0x6B
            ,0x3A ,0x91 ,0x11 ,0x41 ,0x4F ,0x67 ,0xDC ,0xEA ,0x97 ,0xF2 ,0xCF ,0xCE ,0xF0 ,0xB4 ,0xE6 ,0x73
            ,0x96 ,0xAC ,0x74 ,0x22 ,0xE7 ,0xAD ,0x35 ,0x85 ,0xE2 ,0xF9 ,0x37 ,0xE8 ,0x1C ,0x75 ,0xDF ,0x6E
            ,0x47 ,0xF1 ,0x1A ,0x71 ,0x1D ,0x29 ,0xC5 ,0x89 ,0x6F ,0xB7 ,0x62 ,0x0E ,0xAA ,0x18 ,0xBE ,0x1B
            ,0xFC ,0x56 ,0x3E ,0x4B ,0xC6 ,0xD2 ,0x79 ,0x20 ,0x9A ,0xDB ,0xC0 ,0xFE ,0x78 ,0xCD ,0x5A ,0xF4
            ,0x1F ,0xDD ,0xA8 ,0x33 ,0x88 ,0x07 ,0xC7 ,0x31 ,0xB1 ,0x12 ,0x10 ,0x59 ,0x27 ,0x80 ,0xEC ,0x5F
            ,0x60 ,0x51 ,0x7F ,0xA9 ,0x19 ,0xB5 ,0x4A ,0x0D ,0x2D ,0xE5 ,0x7A ,0x9F ,0x93 ,0xC9 ,0x9C ,0xEF
            ,0xA0 ,0xE0 ,0x3B ,0x4D ,0xAE ,0x2A ,0xF5 ,0xB0 ,0xC8 ,0xEB ,0xBB ,0x3C ,0x83 ,0x53 ,0x99 ,0x61
            ,0x17 ,0x2B ,0x04 ,0x7E ,0xBA ,0x77 ,0xD6 ,0x26 ,0xE1 ,0x69 ,0x14 ,0x63 ,0x55 ,0x21 ,0x0C ,0x7D],
            dtype=np.uint8)

def build_hypothesis(measurement: Measurement, byte_idx: int, n_traces: int = 0) -> np.ndarray:
    """
    Build a hypothesis matrix for a single byte of the key by reversing the last round of AES and
    comparing the rounds with the previous one.
    p[i] = i-th measured plaintext
    k[j] = j-th possible key byte
    H[i,j] = sbox[ p[i] xor k[j] ]
    """
    # Load plaintext column
    pt_col = np.loadtxt(measurement.plaintext_path, usecols=byte_idx, converters=hex_to_int, dtype=np.uint8)
    if n_traces != 0:
        pt_col = pt_col[:n_traces]
    # Generate hypothesis matrix
    key_guess = np.arange(256, dtype=np.uint8)
    pt_xor = pt_col[:, np.newaxis] ^ key_guess
    hypothesis_matrix = SBox[pt_xor]

    return hypothesis_matrix

def build_hamming_weight_mtx(hypothesis_matrix: np.ndarray) -> np.ndarray:
    """
    Build a hamming weight matrix for a hypothesis matrix.
    """
    hamming_weight_matrix = np.zeros(hypothesis_matrix.shape, dtype=np.uint8)
    for i in range(hypothesis_matrix.shape[0]):
        for j in range(hypothesis_matrix.shape[1]):
            hamming_weight_matrix[i, j] = bin(hypothesis_matrix[i, j]).count("1")
    return hamming_weight_matrix

def hamm_weight(hex_num : int) -> int:
    """ Calculate the hamming weight of a number """
    return bin(hex_num).count("1")

def hamm_distance(ciphertext_row: np.array, byte_idx: int, keyguess: int):
    byte_idx_shifted = ShiftRowInverse[byte_idx]
    state10 = ciphertext_row[byte_idx_shifted]

    AddRoundKeyByte = keyguess ^ ciphertext_row[byte_idx]
    state9 = SBoxInverse[AddRoundKeyByte]
    # hamming_weight of xorred values is their hamming distance
    return hamm_weight(state9 ^ state10)

def build_hamm_distance_mtx(ct, n_traces: int, byte_idx: int):
    """
    Since bytes will be rearranged within rows, the entire row of ciphertext must be read
    """
    mtx = np.zeros((n_traces, 256))
    # ct_row = ct[trace, :]
    for trace in range(n_traces):
        for keyguess in range(256):
            mtx[trace, keyguess] = hamm_distance(ct[trace, :], byte_idx, keyguess)
    return mtx


def correlate(hamming_mtx: np.ndarray, std_traces_mtx: np.ndarray) -> np.ndarray:
    """
    Build a correlation matrix from a hamming weight matrix (a,b) and a standardized traces matrix.

    Sizes:
    Hamming     : ( measurement_cnt, 256 )
    Traces      : ( measurement_cnt, trace_len )
    Correlation : ( 256, trace_len )

    Note:
    Standardization partially calculates the correlation matrix ( subtracts the mean and divides by the standard deviation ),
    so speeds up the calculation.
    The second matrix is the traces matrix, which is going to be reused for all key bytes, therefore it is standardized beforehand.
    """
    hamming = ((hamming_mtx - np.mean(hamming_mtx, axis=0)) # standardize hamming matrix
                            / np.std(hamming_mtx, axis=0))
    correlation_matrix = ( hamming.T @ std_traces_mtx ) / hamming.shape[0] # complete the correlation calculation
    correlation_matrix = np.abs(correlation_matrix)
    return correlation_matrix


def find_max(correlation_matrix: np.ndarray):
    """ Returns key byte and trace sample (time of leakage) with the maximum correlation."""
    max_in_flattened = np.argmax(correlation_matrix)
    max_indices = np.unravel_index(max_in_flattened, correlation_matrix.shape)
    return max_indices


def build_traces_mtx(measurement: Measurement) -> np.ndarray:
    traces_matrix = (np.fromfile(measurement.trace_path, dtype=np.uint16). # load traces
                     reshape(-1, measurement.trace_length))

    # slice traces matrix to the relevant part
    # traces_matrix = traces_matrix[:, 64:110]
    standardized_traces = ((traces_matrix - np.mean(traces_matrix, axis=0)) # standardize traces to save time
                           / np.std(traces_matrix, axis=0))
    print(f"Full traces mtx shape: {standardized_traces.shape}")
    return standardized_traces

def find_idx_in_arr ( arr, key ):
    for i, element in enumerate(arr):
        if element[0] == key:
            return i

def guessing_entropy(correlation_matrix, processed_byte_idx, correct_key):
        """
        In each row of the correlation matrix (each of the key estimates), the maximum
        and the index of the row where the maximum is located (key estimate) are found.
        These maximum correlations are sorted in descending order, and it is determined
        which one in the sequence is the real key, in terms of the computed correlation.
        """
        # array containing key guess and its correlation
        key_corr_arr = []
        # for each row of correlation matrix find the maximum value and its index
        for (key_guess, row) in enumerate(correlation_matrix):
            # whole row index is a key guess, index of max value
            # within the row is the moment of the leakage
            max_corr_of_guess = np.max(row)
            key_corr_arr.append([key_guess, max_corr_of_guess])
        # sort by correlations, descending
        key_corr_arr.sort(key=lambda x: x[1], reverse=True)
        place_of_correct_key = find_idx_in_arr(key_corr_arr, correct_key[processed_byte_idx])
        print(f"Byte guessing entropy: {place_of_correct_key}/{correlation_matrix.shape[0]}")
        return place_of_correct_key


def find_key(measurement: Measurement, key_length_in_bytes, n_traces: int = 0,
              attack_mode: str = "lrnd", timer: bool = False ) -> Tuple[np.ndarray, str, int]:
    """
    Return the key and its guessing entropy based on the maximum correlation for each byte of the key.
    """
    if attack_mode not in [ "lrnd", "frnd" ]:
        raise ValueError("Unknown attack mode.")

    if timer == True: start_time = time()

    standardized_traces = build_traces_mtx(measurement)
    if n_traces != 0:
        standardized_traces = standardized_traces[:n_traces, :]
    key_arr = np.zeros(key_length_in_bytes, dtype=np.uint8)

    searched_key = measurement.encryption_key
    if attack_mode == "lrnd":
        # ct_mtx = np.loadtxt(measurement.ciphertext_path,
        #                     converters=hex_to_int, dtype=np.uint8)
        ct_mtx = np.fromfile(measurement.ciphertext_path, dtype=np.uint8).reshape(-1, ct_length)

        if n_traces != 0:
            ct_mtx = ct_mtx[:n_traces, :]

        # byte_array = key_schedule(bytes(measurement.encryption_key))[10]
        # int_list = [int(byte) for byte in byte_array]
        searched_key = None # np.array(int_list, dtype=np.uint8)

    # guessing entropies of each subkey byte
    byte_guessing_entropies = []

    for i in range(key_length_in_bytes):
        if attack_mode == "lrnd":
            if n_traces == 0:
                n_traces = measurement.cnt
            hamm_mtx = build_hamm_distance_mtx(ct_mtx, n_traces, i)
        elif attack_mode == "frnd":
            hamm_mtx = build_hamming_weight_mtx(build_hypothesis(measurement, i, n_traces))

        correlation_matrix = correlate(hamm_mtx, standardized_traces)
        key_byte, tracesample_with_max_corr = find_max(correlation_matrix)

        # If the real encryption key is known, calculate the guessing entropy
        if measurement.encryption_key is not None:
            byte_guessing_entropies.append(guessing_entropy(correlation_matrix, i, searched_key))

        print(f"key[{i}]: 0x{key_byte:02X}, sample: {tracesample_with_max_corr}")
        key_arr[i] = key_byte

    if timer == True:
        end_time = time()
        print(f"CPA took: {end_time - start_time:0.0f} seconds")

    # If the real encryption key is known, calculate the guessing entropy
    GE = None
    if measurement.encryption_key is not None:
        GE = np.mean(byte_guessing_entropies)
        print(f"Guessing entropy: {GE:.2f}")

    key_hex_str = ' '.join([hex(i)[2:].zfill(2).upper() for i in key_arr])
    return key_arr, key_hex_str, GE

def verify_key ( measurement: Measurement, key: np.ndarray ) -> bool:
    key_bytes = bytes(key)
    pt = np.loadtxt(measurement.plaintext_path, converters=hex_to_int, dtype=np.uint8)
    ct = np.loadtxt(measurement.ciphertext_path, converters=hex_to_int, dtype=np.uint8)
    pt_bytes = bytes(pt)
    ct_bytes = bytes(ct)

    cipher = AES.new(key_bytes, AES.MODE_ECB)
    ciphertext = cipher.encrypt(pt_bytes)

    return ciphertext == ct_bytes

def red_bg ( text: str ) -> str:
    return Fore.RED + Style.BRIGHT + text + Style.RESET_ALL

def green_bg ( text: str ) -> str:
    return Fore.GREEN + Style.BRIGHT + text + Style.RESET_ALL

def print_key ( found_key: np.ndarray, real_key: np.ndarray = None ) -> bool:
    # the initial encryption key
    print("Found key: ", end='')
    # if real_key is not provided, print only the found key
    if real_key is None:
        print(' '.join([f"0x{byte:02X}" for byte in found_key]))
        return
    for byte in range(len(found_key)):
        keybyte_formatted = f"0x{found_key[byte]:02X}"
        print(
               green_bg(keybyte_formatted)
               if found_key[byte] == real_key[byte]
                else red_bg(keybyte_formatted), end=' '
        )
    print()

def enc_key_from_last_round_key ( key_arr: np.array ) -> np.array:
    encryption_key = reverse_key_schedule(bytes(key_arr), 10)
    return np.frombuffer(encryption_key, dtype=np.uint8)

def cpa(measurement: Measurement, n_traces: int = 0, attack_mode: str = "lrnd", timer: bool = False) -> bool:
    """
    Perform correlation power analysis on given measurement.
    :param Measurement measurement: Traces, PTs, CTs
    :param str attack_mode: lrnd for last round attack, frnd for first round attack
    """
    if n_traces == 0:
        n_traces = measurement.cnt

    match attack_mode:
        case "lrnd":
            print(f"\nPerforming last round CPA using {n_traces} measurements.")
            last_round_key_arr, key_hex, ge = find_key(measurement, measurement.key_length, n_traces=n_traces, timer=True, attack_mode="lrnd")
            key_arr = enc_key_from_last_round_key(last_round_key_arr)
        case "frnd":
            print(f"\nPerforming first round CPA using {n_traces} measurements.")
            key_arr, key_hex, ge = find_key(measurement, measurement.key_length, n_traces=n_traces, timer=True, attack_mode="frnd")
        case _:
            raise ValueError("Unknown attack mode.")

    print("==========================================================================================")
    print_key(key_arr, measurement.encryption_key)

    success = True # verify_key(measurement, key_arr)
    # if success == False and attack_mode == "lrnd":
    #     print("Full last round key wasn't found, even its correct subkeys weren't reversed into correct encryption key subkeys.")
    #     print("Found last round key: ", " ".join([hex(byte)[2:].upper() for byte in last_round_key_arr]))
    # print(f"Attack success: { Fore.GREEN + str(success) if success == True else Fore.RED + str(success) }")
    # print(Style.RESET_ALL, end='')

    return success, ge

def plot_ge_vs_ntraces ( results: List[Tuple[int, float]], trace_cnt, trace_increment_step ):
    import matplotlib.pyplot as plt
    n_traces, ge = zip(*results)
    plt.plot(n_traces, ge, color='red', linewidth=1)
    plt.xlabel("Number of traces")
    plt.ylabel("Guessing entropy")
    plt.title("Guessing entropy vs number of traces")
    plt.xticks(np.arange(0, trace_cnt+trace_increment_step, trace_increment_step))
    plt.grid(True)
    plt.show()


def main():
    WORKING_DIR = (Path.cwd() / 'ni-hsc-lab-02-data').__str__()

    ni_hsc_cpa = Measurement(
        plaintext=None,
        ciphertext=f'{WORKING_DIR}/ciphertext5000.bin',
        trace=f'{WORKING_DIR}/traces5000.bin',
        encryption_key=None
    )

    cpa(ni_hsc_cpa, timer=False, attack_mode="lrnd", n_traces=5000)

if __name__ == "__main__":
    main()
